# **Milestone 02 — Section 2: Data Ingestion, Cleaning & Structural Readiness**
## Commodity price shocks as an early-warning signal for US initial jobless claims

**Course:** Data Science for Social Good  
**Team:** Rana Mohammad Sarib Khan (rk09083, Social Development & Policy, CS minor) · Abdullah Ahmed (aa09303, Computer Engineering) · Hassan Shahzad (ms09070, Computer Science)  
**SDG:** 8 – Decent Work and Economic Growth, Target 8.5

This notebook covers only Section 2 of the Milestone 02 brief: collection audit, alignment, missing-data strategy,
transformations and encoding.
The data pipeline lives in `src/collect.py` (raw pulls) and `src/build_master.py` (weekly master dataset) and is
imported below so the notebook and the scripts can never disagree.

In [1]:
import sys, json, inspect, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from scipy import stats

warnings.filterwarnings("ignore", category=FutureWarning)
ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))
from build_master import (build_master, data_dictionary, weekly_from_daily, add_commodity_features, add_lags,
                          load_daily, load_icsa, COMMODITIES, MAX_LAG, COVID, spike_threshold)

pd.set_option("display.width", 160); pd.set_option("display.max_columns", 40); pd.set_option("display.precision", 3)

# --- plotting conventions used for every figure ---------------------------------------
# Fixed categorical hue per commodity (never re-assigned between figures); claims in near-black ink;
# recessions in neutral gray. Palette validated for colour-vision deficiency (adjacent-pair ΔE ≥ 8).
COLORS = {"oil": "#2a78d6", "gas": "#eb6834", "copper": "#1baf7a", "iron": "#eda100",
          "claims": "#0b0b0b", "rec": "#d9d8d3", "muted": "#52514e"}
LABELS = {"oil": "WTI crude oil", "gas": "Henry Hub natural gas", "copper": "Copper (COMEX)", "iron": "Iron ore 62% Fe"}
UNITS  = {k: v[2] for k, v in COMMODITIES.items()}
mpl.rcParams.update({"figure.dpi": 110, "savefig.dpi": 200, "font.size": 10, "axes.titlesize": 11,
                     "axes.titleweight": "semibold", "axes.labelsize": 10, "axes.spines.top": False,
                     "axes.spines.right": False, "axes.grid": True, "grid.color": "#e6e5e1",
                     "grid.linewidth": 0.6, "axes.edgecolor": "#b5b4ae", "legend.frameon": False})
KEYS = list(COMMODITIES)
print("pandas", pd.__version__, "| project root:", ROOT)

pandas 3.0.6 | project root: /Users/hassan/DS_project


## Build the master dataset
The master table is rebuilt from the cached raw files so that every check below runs against the same object.

In [2]:
master = build_master()
manifest = json.loads((ROOT / "data" / "raw" / "manifest.json").read_text())

overview = pd.DataFrame({
    "Item": ["Observations (rows)", "Variables (columns)", "Observational unit", "Time span",
             "Target variable", "Commodity predictors", "Controls / dummies", "Raw sources"],
    "Value": [f"{master.shape[0]:,} weeks", f"{master.shape[1]}",
              "1 row = 1 calendar week ending Saturday (the US Department of Labor's 'week ending' convention for initial claims)",
              f"{master.index.min().date()} to {master.index.max().date()}",
              "icsa (weekly initial claims, persons) and its derived log / % change / spike indicator",
              "4 commodities × (base weekly mean price, weekly % price shock, 4-week volatility); lags 1–16 of the shock are created on demand by add_lags()",
              "NBER recession (usrec), COVID period (covid_period)",
              "FRED API (6 series), Yahoo Finance via yfinance (2 futures tickers)"]})
display(overview.style.hide(axis="index").set_properties(**{"text-align": "left"}))

sources = (pd.DataFrame(manifest).T
           .loc[:, ["description", "frequency", "rows", "first_date", "last_date", "source"]]
           .rename_axis("series").reset_index())
display(sources.drop(columns="source"))

Item,Value
Observations (rows),"1,500 weeks"
Variables (columns),17
Observational unit,1 row = 1 calendar week ending Saturday (the US Department of Labor's 'week ending' convention for initial claims)
Time span,1998-01-03 to 2026-09-26
Target variable,"icsa (weekly initial claims, persons) and its derived log / % change / spike indicator"
Commodity predictors,"4 commodities × (base weekly mean price, weekly % price shock, 4-week volatility); lags 1–16 of the shock are created on demand by add_lags()"
Controls / dummies,"NBER recession (usrec), COVID period (covid_period)"
Raw sources,"FRED API (6 series), Yahoo Finance via yfinance (2 futures tickers)"


,series,description,frequency,rows,first_date,last_date
0,ICSA,"Initial Claims (weekly, SA, persons)",weekly,1500,1998-01-03,2026-09-26
1,DCOILWTICO,"WTI crude oil spot (daily, USD/bbl)",daily,7499,1998-01-01,2026-09-29
2,DHHNGSP,"Henry Hub natural gas spot (daily, USD/MMBtu)",daily,7499,1998-01-01,2026-09-29
3,USREC,"NBER recession indicator (monthly, 0/1)",monthly,345,1998-01-01,2026-09-01
4,PCOPPUSDM,"Copper global price (monthly, USD/tonne) -- re...",monthly,343,1998-01-01,2026-07-01
5,PIORECRUSDM,"Iron ore global price (monthly, USD/tonne) -- ...",monthly,343,1998-01-01,2026-07-01
6,HG_F,"COMEX copper futures (daily, USD/lb)",daily,6552,2000-08-30,2026-10-02
7,TIO_F,"Iron ore 62% Fe CFR China futures (daily, USD/...",daily,4003,2010-10-14,2026-10-02


# **Section 2: Data Ingestion, Cleaning & Structural Readiness**

## 2.1 Scraping / collection audit
`src/collect.py` pulls every series programmatically and records provenance in `data/raw/manifest.json`.

* **FRED.** Each series is fetched from the keyless CSV endpoint
  `https://fred.stlouisfed.org/graph/fredgraph.csv?id=<SERIES>`. FRED writes a `.` for days with no observation
  (holidays, market closures); these are parsed to `NaN` at load time. One practical finding: FRED stalls on
  browser-style `User-Agent` strings sent without full browser headers, so the request is made with the default
  `python-requests` agent.
* **Yahoo Finance.** `yfinance.download(ticker, start="1998-01-01", interval="1d")` for `HG=F` (COMEX copper,
  USD/lb) and `TIO=F` (iron ore 62% Fe CFR China, USD/tonne). A fallback hits the chart endpoint directly with explicit
  `period1`/`period2` timestamps, because `range=max` silently downgrades the response to monthly bars.
* **Coverage check.** Querying `TIO=F` for 2007–2010 returns `{"error": {"code": "Bad Request",
  "description": "Data doesn't exist for startDate = 1167591600, endDate = 1285873200"}}`: the contract was listed on
  14 October 2010, so iron ore is **structurally missing** before then. Likewise copper futures begin on 30 August
  2000. These gaps are kept as `NaN` rather than back-filled (see 2.3).

In [3]:
# First rows of each raw file, exactly as cached -- the audit trail for the collection step
for name, meta in manifest.items():
    df = pd.read_csv(ROOT / meta["file"], nrows=2)
    print(f"{name:<12} {meta['frequency']:<8} {meta['rows']:>5} rows  {meta['first_date']} -> {meta['last_date']}   "
          f"pulled {meta['pulled_at_utc']}")
    print("   ", df.to_dict("records")[0])

ICSA         weekly    1500 rows  1998-01-03 -> 2026-09-26   pulled 2026-10-04 10:48 UTC
    {'date': '1998-01-03', 'value': 312000}
DCOILWTICO   daily     7499 rows  1998-01-01 -> 2026-09-29   pulled 2026-10-04 10:48 UTC
    {'date': '1998-01-01', 'value': nan}
DHHNGSP      daily     7499 rows  1998-01-01 -> 2026-09-29   pulled 2026-10-04 10:48 UTC
    {'date': '1998-01-01', 'value': nan}
USREC        monthly    345 rows  1998-01-01 -> 2026-09-01   pulled 2026-10-04 10:48 UTC
    {'date': '1998-01-01', 'value': 0}
PCOPPUSDM    monthly    343 rows  1998-01-01 -> 2026-07-01   pulled 2026-10-04 10:48 UTC
    {'date': '1998-01-01', 'value': 1687.60000986328}
PIORECRUSDM  monthly    343 rows  1998-01-01 -> 2026-07-01   pulled 2026-10-04 10:48 UTC
    {'date': '1998-01-01', 'value': 13.41}
HG_F         daily     6552 rows  2000-08-30 -> 2026-10-02   pulled 2026-10-04 10:48 UTC
    {'date': '2000-08-30', 'open': 0.8790000081062317, 'high': 0.8870000243186951, 'low': 0.8769999742507935, 'clos

## 2.2 Data alignment and structural organisation
The raw files arrive at three frequencies (weekly, daily, monthly) and two different week conventions.
The reshaping below produces one modelling-ready table keyed on the claims reference week:

1. **Daily → weekly (down-sampling).** Each daily price series is grouped into Sunday–Saturday bins labelled by
   the Saturday (`resample("W-SAT")`). Per week we keep the *mean* price (the base weekly price) and a trailing
   20-trading-day volatility of daily log returns sampled on the last day of the week; weeks with no trading day at
   all are set to missing.
2. **Monthly → weekly (up-sampling).** The NBER recession flag is forward-filled from the month start to every week
   in that month.
3. **Join.** All weekly frames are left-joined onto the claims index, so the claims weeks define the sample and
   no commodity week without a claims observation enters the table.
4. **Feature construction.** Week-over-week percentage change of the weekly mean (the price shock). Lags 1–16 of
   the shock, the proposal's exploratory window, are produced on demand by `add_lags()` rather than stored.

The functions that do this are printed below from `src/build_master.py`.

In [4]:
print(inspect.getsource(weekly_from_daily))
print(inspect.getsource(add_commodity_features))
print(inspect.getsource(add_lags))

def weekly_from_daily(daily: pd.Series, key: str) -> pd.DataFrame:
    """Collapse a daily price series to Sunday-Saturday weeks (label = Saturday).

    Returns, per week:
        {key}_mean    mean of daily prices in the week      (base weekly price)
        {key}_vol_4w  std of daily log returns over the trailing 20 trading days,
                      taken at the last day of the week      (4-week rolling volatility)
    Weeks with no trading day at all (exchange closures) are set to NaN.
    """
    # WTI printed a negative settlement on 2020-04-20 (-37.63 USD/bbl); the log
    # return is undefined there, so that single day is excluded from the
    # return-based volatility feature only. Price levels keep the true value.
    logret = np.log(daily.where(daily > 0)).diff()
    vol20 = logret.rolling(20, min_periods=10).std()
    wk = pd.DataFrame({
        f"{key}_mean":   daily.resample("W-SAT").mean(),
        f"{key}_vol_4w": vol20.resample("W-SAT").last(),
    })
    n_days = da

In [5]:
# Worked example of the daily -> weekly collapse for one week, so the aggregation is auditable.
# The week ending 2020-04-25 contains the only negative WTI settlement in history (2020-04-20).
oil_daily = load_daily("oil")
example_week = oil_daily["2020-04-19":"2020-04-25"]
print("Daily WTI prices, week ending 2020-04-25 (USD/barrel):")
print(example_week.to_string())
print("\nWeekly row produced for that week:")
display(master.loc[["2020-04-25"], ["oil_mean", "oil_pct_1w", "oil_vol_4w"]])

# Alignment check: every master index date is a Saturday and spacing is exactly 7 days
print("All Saturdays:", (master.index.dayofweek == 5).all(),
      "| unique 7-day spacing:", master.index.to_series().diff().dropna().unique())

Daily WTI prices, week ending 2020-04-25 (USD/barrel):
date
2020-04-20   -36.98
2020-04-21     8.91
2020-04-22    13.64
2020-04-23    15.06
2020-04-24    15.99

Weekly row produced for that week:


,oil_mean,oil_pct_1w,oil_vol_4w
week_ending,,,
2020-04-25,3.324,-83.479,0.159


All Saturdays: True | unique 7-day spacing: <TimedeltaArray>
['7 days']
Length: 1, dtype: timedelta64[us]


### Interpretation
The worked example shows the collapse from five trading days to a single row and makes the choice of
*mean* rather than *close* visible: the week of 25 April 2020 closes at 15.99 USD/bbl, but its mean is 3.32 USD/bbl
because of the −37.63 USD/bbl print on 20 April. The weekly mean is the stored level feature because it is what a
business paying for inputs over the week actually faced; the close is not kept. The log return
on the negative day is undefined, so that single observation is excluded from the return-based volatility
features only, never from the price level. The index passes both structural checks (all Saturdays, uniform 7-day spacing).

## 2.3 Cleaning and missing-data strategy

In [6]:
miss = pd.DataFrame({"missing_n": master.isna().sum(), "missing_pct": master.isna().mean() * 100})
miss["missing_pct"] = miss["missing_pct"].round(2)
display(miss.T)

# where is the missingness in time?  share of weeks per year with a missing weekly mean
by_year = (master[[f"{k}_mean" for k in KEYS]].isna()
           .groupby(master.index.year).mean().mul(100).round(0).astype(int))
by_year.columns = KEYS
by_year.index.name = "year"
display(by_year.T)

,icsa,log_icsa,icsa_pct_chg_1w,oil_mean,oil_pct_1w,oil_vol_4w,gas_mean,gas_pct_1w,gas_vol_4w,copper_mean,copper_pct_1w,copper_vol_4w,iron_mean,iron_pct_1w,iron_vol_4w,usrec,covid_period
missing_n,0.0,0.0,1.00,4.00,8.00,6.0,121.00,203.00,126.0,139.00,140.00,141.0,667.00,668.00,669.0,0.0,0.0
missing_pct,0.0,0.0,0.07,0.27,0.53,0.4,8.07,13.53,8.4,9.27,9.33,9.4,44.47,44.53,44.6,0.0,0.0


year,1998,1999,2000,2001,2002,2003,2004,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025,2026
oil,2,6,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
gas,17,25,13,12,29,37,35,21,31,13,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
copper,100,100,66,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
iron,100,100,100,100,100,100,100,100,100,100,100,100,79,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


### Missing-data findings and handling rationale

| Variable group | Missing | Mechanism | Decision |
|---|---|---|---|
| `icsa` and all target columns | 0 % (1 % for the first-difference, by construction) | — | Never imputed. |
| `oil_*` | 0.3 % (4 weeks in 1998–99) | Sparse early FRED daily coverage | Left `NaN`. |
| `gas_*` | 8.1 % (121 weeks, all before April 2007) | FRED's Henry Hub daily series is intermittent before 2007 (roughly one trading day in four is recorded) | Left `NaN`; documented as source sparsity. |
| `copper_*` | 9.3 % (all weeks before 2000-09-02) | **Structural:** COMEX copper futures history on Yahoo begins 30 Aug 2000 | Left `NaN`. |
| `iron_*` | 44.5 % (all weeks before 2010-10-16) | **Structural:** TIO=F listed 14 Oct 2010; the API returns "Data doesn't exist" for earlier dates | Left `NaN`. |
| `*_vol_4w` | a few extra weeks per commodity | The 20-day rolling window needs ≥ 10 returns | Left `NaN`. |
| `usrec`, `covid_period` | 0 % | — | — |

**Why no imputation.** Mean/median imputation of a price level that did not exist would inject a constant into a
time series and destroy the very week-to-week variation the project studies; time-series interpolation across a
*ten-year* hole (iron ore) would be fabrication, not estimation. Back-filling iron ore from FRED's monthly benchmark
was considered and rejected on two grounds: (i) before 2010 iron ore was priced on *annual* contracts, so the FRED
series is a flat step function (36.63 USD/t for all of 2007) carrying no weekly information, and (ii) mixing two
sources in one column contradicts the single-source decision made in Milestone 01. The target is complete, so no
rows are dropped from the master table; instead, each model will use **listwise deletion on the columns it actually
uses**. A model with copper will therefore train on 2000–2026 (1,361 weeks) and a model with iron ore on 2010–2026
(833 weeks).

**Selection bias.** Listwise deletion is harmless only if missingness is unrelated to the outcome. Here it is
related to *time*, which is in turn related to the outcome: any specification that includes iron ore sees neither
the 2001 recession nor the 2008–09 financial crisis, the two largest claims episodes other than COVID-19. Such a
model would be estimated on a sample with a narrower range of labour-market stress and could understate effects
that only materialise in deep recessions. We flag this explicitly and plan to report metal-based models both with
and without iron ore. The pre-2007 gas sparsity has the opposite character: missing weeks are scattered across
ordinary years, so their omission is closer to random.

## 2.4 Transformations and encoding

In [7]:
# --- Feature rescaling: log transform for heavy right skew ---------------------------------
skew_tbl = pd.DataFrame({
    "variable": ["icsa"] + [f"{k}_mean" for k in KEYS],
    "skew (raw)": [stats.skew(master["icsa"])] + [stats.skew(master[f"{k}_mean"].dropna()) for k in KEYS],
    "skew (log)": [stats.skew(master["log_icsa"])] + [stats.skew(np.log(master[f"{k}_mean"].dropna())) for k in KEYS],
}).round(2)
display(skew_tbl)

,variable,skew (raw),skew (log)
0,icsa,11.22,2.30
1,oil_mean,0.11,-0.90
2,gas_mean,1.75,0.55
3,copper_mean,0.01,-1.08
4,iron_mean,0.47,-0.34


### Interpretation: log transformation
Initial claims are extremely right-skewed (skewness 11.2): only 50 of 1,500 weeks exceed 700,000, but the
pandemic weeks reach 6.1 million. `log_icsa` reduces skewness to 2.3 and is the response scale we intend to model;
a one-unit change in log claims is a constant *percentage* change, which is also how policymakers talk about
layoff waves. Natural gas is the only commodity with material right skew (1.75, driven by the 2005 and 2022 spikes),
and the log reduces it to 0.55. Crude oil, copper and iron ore are already close to symmetric in levels (0.11,
0.01, 0.47) and the log *over*-corrects oil and copper into moderate left skew (−0.9, −1.1), so for the commodities
the raw level is what the master stores; a log can be applied at model time if an elasticity specification is wanted.

In [8]:
# --- Scaling: z-score standardisation and Min-Max normalisation -------------------------
scale_cols = ["icsa", "log_icsa", "oil_mean", "gas_mean", "copper_mean", "iron_mean"]
raw = master[scale_cols]
z_scored = (raw - raw.mean()) / raw.std(ddof=1)                 # standardisation: mean 0, sd 1
min_max = (raw - raw.min()) / (raw.max() - raw.min())            # normalisation: range [0, 1]
scale_demo = pd.concat({
    "raw":    raw.describe().loc[["mean", "std", "min", "max"]],
    "z":      z_scored.describe().loc[["mean", "std", "min", "max"]],
    "minmax": min_max.describe().loc[["mean", "std", "min", "max"]],
}, axis=0).round(3)
display(scale_demo)

icsa  log_icsa  oil_mean  gas_mean  copper_mean  iron_mean
raw    mean  3.616e+05    12.689    61.120     4.061     6462.296    106.807
       std   3.307e+05     0.382    26.925     2.152     2823.633     36.174
       min   1.890e+05    12.150     3.324     1.400     1359.148     38.730
       max   6.137e+06    15.630   142.518    14.820    14787.709    218.396
z      mean -0.000e+00    -0.000    -0.000    -0.000       -0.000     -0.000
       std   1.000e+00     1.000     1.000     1.000        1.000      1.000
       min  -5.220e-01    -1.413    -2.147    -1.237       -1.807     -1.882
       max   1.747e+01     7.708     3.023     5.000        2.948      3.085
minmax mean  2.900e-02     0.155     0.415     0.198        0.380      0.379
       std   5.600e-02     0.110     0.193     0.160        0.210      0.201
       min   0.000e+00     0.000     0.000     0.000        0.000      0.000
       max   1.000e+00     1.000     1.000     1.000        1.000      1.000

### Interpretation: scaling
Both scalings are computed here and deliberately **not stored** in the master. Standardisation is the default for the
regression-type models planned (coefficients become comparable across commodities measured in dollars per barrel,
per MMBtu and per tonne); Min-Max normalisation serves bounded-input models. Because the min, max, mean and standard
deviation above are **full-sample** statistics, writing the scaled columns to disk would bake look-ahead information
into the file; in Milestone 03 the scalers are fitted on the training window inside the model pipeline.

In [9]:
# --- Categorical encoding: nominal strings -> Boolean dummy indicator flags ----------------
# Two nominal variables are constructed from the data and encoded with pandas.get_dummies:
nominal = pd.DataFrame({
    "quarter": "Q" + master.index.quarter.astype(str),
    "regime":  np.where(master["usrec"] == 1, "recession", "expansion"),
}, index=master.index)
print("Nominal string columns (first 3 rows):"); display(nominal.head(3))

dummies = pd.get_dummies(nominal, prefix=["q", "regime"], drop_first=True, dtype=int)  # reference: Q1, expansion
print("\nAfter get_dummies(drop_first=True) -> one Boolean 0/1 flag per non-reference level:")
display(dummies.head(3))

# The master's usrec column is exactly this regime dummy; verify
assert (dummies["regime_recession"].to_numpy() == master["usrec"].to_numpy()).all()

dummy_counts = master[["usrec", "covid_period"]].agg(["sum", "mean"]).T
dummy_counts.columns = ["weeks = 1", "share"]
dummy_counts["share"] = dummy_counts["share"].round(3)
display(dummy_counts)

Nominal string columns (first 3 rows):


,quarter,regime
week_ending,,
1998-01-03,Q1,expansion
1998-01-10,Q1,expansion
1998-01-17,Q1,expansion



After get_dummies(drop_first=True) -> one Boolean 0/1 flag per non-reference level:


,q_Q2,q_Q3,q_Q4,regime_recession
week_ending,,,,
1998-01-03,0,0,0,0
1998-01-10,0,0,0,0
1998-01-17,0,0,0,0


,weeks = 1,share
usrec,120.0,0.080
covid_period,43.0,0.029


### Interpretation: dummy encoding
`get_dummies(drop_first=True)` converts the two nominal strings into four 0/1 indicators (`Q2`, `Q3`, `Q4`,
`recession`); Q1 and expansion are the reference levels, which avoids the dummy-variable trap in a regression with
an intercept. Only the regime dummy is kept in the master, as `usrec` (the assertion confirms the two are identical).
The quarter dummies are shown for the encoding mechanics but not stored, because initial claims are published
seasonally adjusted and a quarter effect would be double-counting. The other stored flag is `covid_period`
(43 weeks), which the proposal anticipates needing to isolate the pandemic layoffs. Only 120 weeks (8 %) fall inside
an NBER recession, which already warns that any regime-interaction hypothesis will rest on a small number of
recession observations.